# The Python System Interpreter

*One Ubuntu server, two colleagues, three Python projects.*

![Image Description](project_logo.png)

## Act I — The Ticket

### The Cast

| Person | Role | Linux user | `sudo` |
| --- | --- | --- | --- |
| **Alice** | Application developer. Ships two Python web services and wants to move fast. | `alice` | no |
| **Bob** | Server admin. Fifteen years of "one machine, one Python", and no reason to change. | `bob` | yes |

### Scene 1 — Alice Asks for a Server

> **Alice:** Hi Bob. I'm starting two Python projects. Could you give me a Linux box?
>
> **Bob:** Sure, Ubuntu 24.04. It ships with Python, so you're good to go.
>
> **Alice:** Thanks. The two projects need different versions of the same libraries, though.
>
> **Bob:** Different versions? On one server? No. Tell me what you need and I'll install it.
>
> **Alice:** ...let's see how that goes.

Bob's mental model is the one Linux itself suggests: **one** interpreter at `/usr/bin/python3`, shared by everyone. It works perfectly until two programs need *different versions of the same library* — and then it fails quietly.

We hit three walls and knock each one down:

1. **Shared machine.** The library version Bob pins breaks Alice's project.
2. **Shared user.** `pip install --user` separates people, but not Alice's two projects.
3. **Virtual environments.** Every project gets its own packages, and all three applications coexist.

### Scene 2 — Bob Provisions the Server

> **Bob:** Here you go. Fresh Ubuntu 24.04, two accounts, Python installed. Don't break anything.

The "server" in this story is an **OCI container image** built from [.devcontainer/Dockerfile](.devcontainer/Dockerfile) — a packaged, reproducible copy of a machine's file system, so everyone gets the same Ubuntu, the same interpreter and the same accounts.

Bob's base image holds nothing but the essentials:

| Installed | Why |
| --- | --- |
| `python3`, `python3-dev` | The shared system interpreter, plus the headers packages need when they compile C code. |
| `python3-pip` | Python's package installer. |
| `python3-venv` | Creates isolated environments. Bob has it and never uses it. |
| `notebook`, `jupyterlab`, `ipykernel` | So this lesson can run inside the server. |
| the users `bob` and `alice` | Real accounts, so ownership and permissions behave realistically. |

What is **not** on the image: any application dependency. No `requests`, no `fastapi`, no `pydantic`, no monitoring helper. Those get installed later, by the right person, in the order the story needs — because that order is what produces the collisions.

---

## Act II — Taking Inventory of the New Server

### The Machine and Its Users

#### Which Distribution?

> **Alice:** Before I install anything, I want to see what I actually got.

`uname -s` prints the kernel name. `/etc/os-release` is the file every Linux distribution ships to describe itself; sourcing it with `.` makes `$PRETTY_NAME` available.

In [ ]:
%%bash
echo "Platform: $(uname -s) | Distro: $(. /etc/os-release && echo "$PRETTY_NAME")"

#### Which Accounts Exist?

`getent passwd <name>` asks the system's user database for one account; `awk -F:` cuts the colon-separated record into readable fields.

In [2]:
%%bash
echo "-- User Information --"
echo "Bob:"
getent passwd bob | awk -F: '{print "user="$1, "uid="$3}'
echo ""
echo "Alice:"
getent passwd alice | awk -F: '{print "user="$1, "uid="$3}'

-- User Information --
Bob:
user=bob uid=1000

Alice:
user=alice uid=1001


#### Who Is Running This Notebook?

The dev container logs in as `bob`, so every command runs with his identity unless we switch users on purpose. `whoami` settles it.

In [5]:
%%bash
echo "This Jupyter Notebook runs as '$(whoami)'"

This Jupyter Notebook runs as 'bob'


Alice's account exists too, but she is not at the keyboard. Her record also shows her home directory — the place where her private package folder will appear later.

In [ ]:
%%bash
getent passwd alice | awk -F: '{print "The user " $1 " (uid=" $3 ") exists, with the home directory " $6}'

### The Shared Interpreter

#### Which `python3`?

`which python3` answers "which program starts when somebody types `python3`?". There is exactly one answer, and it is the same for Bob and for Alice. That binary is the **system interpreter**: not Bob's Python, not Alice's Python — the *machine's*.

In [9]:
%%bash
echo "This Jupyter Notebook uses '$(which python3)' and '$(python3 --version)' as Python version"

This Jupyter Notebook uses '/usr/bin/python3' and 'Python 3.10.12' as Python version


#### Where Does It Look for Packages?

`python3 -m site` shows the three things that decide every conflict in this notebook:

- **`sys.path`** — the ordered list of directories scanned on `import`. The first match wins, so the order decides which copy of a library is used.
- **`USER_SITE`** — a private per-user directory (`~/.local/lib/python3.X/site-packages`). Probably missing for now; pip creates it on the first `--user` install.
- **`ENABLE_USER_SITE`** — whether that private directory counts at all.

Bob first.

In [10]:
%%bash
python3 -m site

sys.path = [
    '/workspace',
    '/usr/lib/python310.zip',
    '/usr/lib/python3.10',
    '/usr/lib/python3.10/lib-dynload',
    '/usr/local/lib/python3.10/dist-packages',
    '/usr/lib/python3/dist-packages',
]
USER_BASE: '/home/bob/.local' (doesn't exist)
USER_SITE: '/home/bob/.local/lib/python3.10/site-packages' (doesn't exist)
ENABLE_USER_SITE: True


Now Alice. The interpreter path is identical — the same file on disk — and so are the system directories. Only `USER_SITE` differs, pointing into `/home/alice`.

That single line is the *entire* isolation the operating system gives us for free, and it will not be enough.

In [11]:
%%bash
sudo su - alice -c "python3 -m site"

sys.path = [
    '/home/alice',
    '/usr/lib/python310.zip',
    '/usr/lib/python3.10',
    '/usr/lib/python3.10/lib-dynload',
    '/home/alice/.local/lib/python3.10/site-packages',
    '/usr/local/lib/python3.10/dist-packages',
    '/usr/lib/python3/dist-packages',
]
USER_BASE: '/home/alice/.local' (exists)
USER_SITE: '/home/alice/.local/lib/python3.10/site-packages' (exists)
ENABLE_USER_SITE: True


## Act III — Everybody Installs Their Dependencies

### Bob Installs a System Package with APT

#### The Package: `python3-psutil`

> **Bob:** My nightly script has to report how the box is doing — CPU, memory, uptime. I'll take that from the distribution, like a normal person.

Bob reaches for **APT**, Ubuntu's package manager, and here he is genuinely right: APT packages are built for this exact Ubuntu release, patched by the regular security updates, and available to every user once installed.

He wants **`python3-psutil`** — *process and system utilities* — the standard library for reading what `top`, `free` and `uptime` show: CPU load, memory usage, disks, processes, boot time. A perfect fit for a health check, and correctly delivered by the distribution.

> 💡 **Why not `python3-systemd`?** It can only talk to a running `systemd-journald`, which a container does not have: the import succeeds and every log line silently vanishes. `psutil` just reads `/proc`, which is always there.

`apt-get update` refreshes the catalogue of available packages and `apt-get install -y` installs without asking. Both need `sudo` — which is exactly why Alice cannot do this herself.

In [ ]:
%%bash
set -euo pipefail
sudo apt-get update
sudo apt-get install -y --no-install-recommends python3-psutil

#### Verify the Installation

Bob asks APT what it believes is installed. `apt list --installed` prints thousands of lines, so `grep` keeps only the interesting one.

Note the version: it is the one Ubuntu chose for this release. Stability is the deal APT offers, and giving up version choice is the price.

In [ ]:
%%bash
apt list --installed 2>/dev/null | grep python3-psutil

python3-systemd/now 234-3ubuntu2 amd64 [installed,local]


The same package, seen from Python's side. `python3 -m pip list` shows every distribution the interpreter can import, no matter who installed it, so the APT package appears here under its plain name `psutil`.

Two package managers, one import path, neither aware of the other — a reliable source of confusion on shared machines.

In [ ]:
%%bash
python3 -m pip list 2>/dev/null | grep -i psutil

systemd-python            234


#### Where the Files Land

APT-managed Python packages go to `/usr/lib/python3/dist-packages/`. Everything inside belongs to a `.deb` and is owned by `root`. Alice can read it — it is on her import path too — but she can never write to it.

In [ ]:
%%bash
DIST_PACKAGES="/usr/lib/python3/dist-packages"

echo "Directory : ${DIST_PACKAGES}"
echo "Owner     : $(stat -c '%U:%G' "${DIST_PACKAGES}") with permissions $(stat -c '%a' "${DIST_PACKAGES}")"
echo "Entries   : $(ls -1 "${DIST_PACKAGES}" | wc -l)"
echo ""
echo "Everything psutil placed there:"
ls -1 "${DIST_PACKAGES}" | grep -i psutil

total 48K
drwxr-xr-x 11 root root 4.0K Sep  1 10:47 .
drwxr-xr-x  3 root root 4.0K Sep  1 10:46 ..
drwxr-xr-x  3 root root 4.0K Sep  1 10:47 _distutils_hack
drwxr-xr-x  5 root root 4.0K Sep  1 10:47 pip
drwxr-xr-x  2 root root 4.0K Sep  1 10:47 pip-22.0.2.dist-info
drwxr-xr-x  6 root root 4.0K Sep  1 10:47 pkg_resources
drwxr-xr-x  7 root root 4.0K Sep  1 10:47 setuptools
drwxr-xr-x  2 root root 4.0K Sep  1 10:47 setuptools-59.6.0.egg-info
drwxr-xr-x  4 root root 4.0K Sep  1 10:47 systemd
-rw-r--r--  1 root root  586 Mar 17  2022 systemd_python-234.egg-info
drwxr-xr-x  5 root root 4.0K Sep  1 10:47 wheel
drwxr-xr-x  2 root root 4.0K Sep  1 10:47 wheel-0.37.1.egg-info


### Bob Pins His Library Version System-Wide

#### The Install

> **Bob:** My legacy tool has run on `requests` 2.25.1 since forever. I don't touch a working system, so that exact version goes on the server.

`requests` is the classic HTTP library. Bob *pins* 2.25.1 — one exact release instead of "whatever is newest" — because an unattended upgrade broke him once.

The pin is sound engineering. **Where** he puts it is not: `sudo python3 -m pip install` writes into `/usr/local/lib/python3.X/dist-packages/`, which every account on this machine imports from. He thinks he is choosing a version for his application; he is choosing it for everyone who will ever log in.

Two flags to know:

- **`-H`** switches `HOME` to root's, so pip does not scatter cache files into Bob's home.
- **`--break-system-packages`** is Ubuntu 24.04 asking "are you sure?". Since PEP 668, pip refuses to modify a distribution-managed interpreter without it. It is a warning label, not a workaround.

> **Bob:** Whatever. It's my server.

In [ ]:
%%bash
set -euo pipefail
echo "Bob pins requests 2.25.1 -- for the whole machine"
sudo -H python3 -m pip install --break-system-packages "requests==2.25.1"

#### Who Else Just Got It?

Bob verifies his install. The version below is what *any* `import requests` on this machine now resolves to.

In [ ]:
%%bash
echo "Packages visible to '$(whoami)':"
python3 -m pip list 2>/dev/null | grep -i requests

requests                  2.25.1


Here the trouble becomes visible — though nobody notices yet. Alice never asked for `requests`, never installed it and cannot change it. The same command run as her prints the same 2.25.1, because the package sits in a directory her interpreter reads too.

A decision made in Bob's shell has silently become a fact in Alice's.

In [ ]:
%%bash
echo "Packages visible to 'alice':"
sudo -u alice -H python3 -m pip list 2>/dev/null | grep -i requests

requests                  2.32.3


For completeness: pip installs made by the local administrator land in `/usr/local/lib/pythonX.Y/dist-packages/` — next to, but strictly separate from, the APT directory, so that `apt upgrade` and `sudo pip install` cannot overwrite each other. Different folders, same consequence: both are on everyone's import path.

In [17]:
%%bash
LOCAL_ADMIN_SITE=$(python3 -c "import sys; print(f'/usr/local/lib/python{sys.version_info.major}.{sys.version_info.minor}/dist-packages')")
echo "$LOCAL_ADMIN_SITE"
ls -lah "$LOCAL_ADMIN_SITE" | grep requests

/usr/local/lib/python3.10/dist-packages
drwxr-xr-x  3 root root 4.0K Sep  1 11:22 requests
drwxr-xr-x  2 root root 4.0K Sep  1 11:22 requests-2.25.1.dist-info


### Alice Installs Into Her Own User Site

#### The Install

> **Bob:** Server's built, tickets are closed. I'm getting a coffee — that was a lot of typing for one morning.
>
> **Alice:** Enjoy it. I'll take it from here.

Alice has no `sudo`, and does not want it. She needs **FastAPI** (a framework for building web APIs) and **Pydantic** (the data-validation library FastAPI is built on). Those are *application* dependencies, and they have no business in the operating system.

So she uses `--user`, which installs into `~/.local/lib/python3.X/site-packages` — a folder she owns. Python puts it on the import path automatically, **before** the system directories, so her packages win for her and stay invisible to everyone else.

She pins `fastapi==0.68.2`, because her first project was written against it, and `pydantic<2`, because that release predates Pydantic's v2 rewrite. `--break-system-packages` is needed even here: Ubuntu guards the whole interpreter, user site included, although nothing outside `/home/alice` is touched.

In [ ]:
%%bash
set -euo pipefail
echo "Alice installs her framework into her own user site"
sudo -u alice -H python3 -m pip install --user --break-system-packages \
    "fastapi==0.68.2" "pydantic<2"

#### Only Alice Can See It

Adding `--user` to `pip list` restricts the listing to the user site. As Bob it comes back empty — everything he uses is system-wide. As Alice it shows her FastAPI and Pydantic.

One interpreter, two answers, decided by nothing but who is asking. This is the first real isolation boundary — and notice how narrow it is.

In [ ]:
%%bash
echo "The user '$(whoami)' has the following packages in their user site:"
python3 -m pip list --user 2>/dev/null || echo "  (nothing)"

echo ""
echo "The user 'alice' has the following packages in their user site:"
sudo -u alice -H python3 -m pip list --user 2>/dev/null

The User 'bob' has the following Python packages installed in their user site:


## Act IV — The Three Applications

Three applications now have to live on this one server, each a single `main.py` so that nothing distracts from the dependencies they declare:

| Application | Owner | `requests` | `pydantic` | `fastapi` |
| --- | --- | --- | --- | --- |
| Bob's Legacy Application | `bob` | **exactly 2.25.1** | – | – |
| Alice FastAPI v1 | `alice` | 2.27 or newer | **v1** | 0.68.2 |
| Alice FastAPI v2 | `alice` | 2.27 or newer | **v2** | 0.111.1 |

> ⚠️ **Two irreconcilable requirements**
>
> **Bob versus Alice.** `requests.exceptions.JSONDecodeError` arrived in requests 2.27; Bob pinned 2.25.1 for the whole machine. No single version satisfies both.
>
> **Alice versus Alice.** Project one imports `BaseSettings`, removed in Pydantic v2. Project two calls `model_dump()`, which exists only in Pydantic v2. No single version satisfies both either.

Nobody wrote bad code. These conflicts can only be solved by making sure the three applications never have to agree.

### Bob's Legacy Application

> **Bob:** It's not fancy, but it has run every night since 2014.

Bob's tool does two things. It takes a **health snapshot** with `psutil` — CPU load, memory in use, uptime — and it **watches a domain** by building an HTTP request with `requests` 2.25.1. It stops at building the request rather than sending it, so the notebook stays offline-friendly.

Everything is reported through Python's built-in `logging` module: standard library, identical inside and outside a container, and redirectable to the journal, a file or syslog by changing configuration instead of imports.

The cell below is shell, not Python. It creates the project directory, writes `main.py` with a *here-document* (`<<'PYTHON' ... PYTHON` copies the text verbatim), fixes ownership, and prints the result.

In [ ]:
%%bash
set -euo pipefail

BOB_LEGACY="/home/bob/projects/legacy-project"
BOB_MAIN="${BOB_LEGACY}/main.py"

mkdir -p "${BOB_LEGACY}"

cat > "${BOB_MAIN}" <<'PYTHON'
"""Bob's legacy application: nightly host health and domain watch."""
import argparse
import datetime
import logging
import sys

import psutil
import requests

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-5s bob-legacy: %(message)s",
    stream=sys.stdout,
)
log = logging.getLogger("bob-legacy")


def host_health():
    """Read the same numbers that 'top', 'free' and 'uptime' would show."""
    booted_at = datetime.datetime.fromtimestamp(psutil.boot_time())
    uptime = datetime.datetime.now() - booted_at
    return {
        "cpu_percent": psutil.cpu_percent(interval=0.1),
        "memory_percent": psutil.virtual_memory().percent,
        "uptime": str(uptime).split(".")[0],
    }


def create_request(domain):
    """Prepare the request used by the legacy application."""
    return requests.Request(method="GET", url=domain)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--domain", default="https://example.com")
    args = parser.parse_args()

    log.info("Bob Legacy Application")
    log.info("  python   : %s", sys.executable)
    log.info("  psutil   : %s", psutil.__version__)
    log.info("  requests : %s", requests.__version__)

    for name, value in host_health().items():
        log.info("  host.%-14s: %s", name, value)

    request = create_request(args.domain)
    log.info("  watching : %s %s", request.method, request.url)


if __name__ == "__main__":
    main()
PYTHON

chown -R bob:bob "${BOB_LEGACY}"

#
# Show the content of main.py
cat "${BOB_MAIN}"

# Bob's legacy application.
import argparse
import sys

import requests


def create_request(domain):
    """Prepare the request used by the legacy application."""
    return requests.Request(method="GET", url=domain)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--domain", default="https://example.com")
    args = parser.parse_args()

    print("Bob Legacy Application")
    print(f"  Python  : {sys.executable}")
    print(f"  requests: {requests.__version__}")

    request = create_request(args.domain)
    print(f"  request : {request.method} {request.url}")


if __name__ == "__main__":
    main()


Bob runs it the only way he knows: he hands the file to the system interpreter. Nothing activated, nothing sourced. `python3 main.py` finds `psutil` in the APT directory and `requests` in the system-wide pip directory, and it works.

For Bob the machine is finished. That is why the coming bug survives so long in real organisations: from the admin's seat, everything looks perfect.

In [23]:
%%bash
set -euo pipefail
python3 /home/bob/projects/legacy-project/main.py --domain "https://example.com"

Bob Legacy Application
  Python  : /usr/bin/python3
  requests: 2.25.1
  request : GET https://example.com


### Alice FastAPI v1

A small **FastAPI** service. FastAPI turns ordinary functions into an HTTP API: `@app.get("/")` means "on a request to the root URL, call this function and return its result as JSON".

Two details decide the rest of the story:

- She configures the service with `BaseSettings` from **Pydantic**. In v1 that class lives in the main package; in v2 it moved into a separate `pydantic-settings` distribution. Her project therefore needs **Pydantic v1**.
- She catches `requests.exceptions.JSONDecodeError`, so a backend returning broken JSON produces a friendly message instead of a stack trace.

> ⚠️ **The first collision.** `JSONDecodeError` arrived in **requests 2.27**; Bob installed **2.25.1** machine-wide. Her code is correct, his pin is defensible, and the import fails anyway — purely because they share one copy of one library.

The cell writes her files with `sudo tee` and hands ownership straight back with `chown`, so the result matches what she would have typed in her own shell.

Here is the code that gets written to `/home/alice/projects/fastapi-v1/main.py`.

In [ ]:
%%bash
set -euo pipefail

ALICE_V1="/home/alice/projects/fastapi-v1"
ALICE_MAIN="${ALICE_V1}/main.py"

sudo mkdir -p "${ALICE_V1}"

sudo tee "${ALICE_MAIN}" > /dev/null <<'PYTHON'
# Alice FastAPI project 1
#   requires pydantic v1  -> BaseSettings lives in the main package
#   requires requests>=2.27 -> requests.exceptions.JSONDecodeError
import sys

import fastapi
import pydantic
import requests
from fastapi import FastAPI

# removed from pydantic v2, moved to the separate "pydantic-settings" package
from pydantic import BaseSettings

# unsupported by "requests==2.25.1"
from requests.exceptions import JSONDecodeError


class Settings(BaseSettings):
    app_name: str = "alice-fastapi-v1"


settings = Settings()
app = FastAPI(title=settings.app_name)


def parse_response(response):
    try:
        return response.json()
    except JSONDecodeError:
        return {"error": "The service did not return JSON"}


@app.get("/")
def root():
    response = requests.Response()
    response._content = b"not valid JSON"

    return {
        "project": settings.app_name,
        "fastapi": fastapi.__version__,
        "pydantic": pydantic.VERSION,
        "requests": requests.__version__,
        "parsed_response": parse_response(response),
    }


if __name__ == "__main__":
    print("Alice FastAPI v1")
    print(f"  Python  : {sys.executable}")
    print(f"  fastapi : {fastapi.__version__}")
    print(f"  pydantic: {pydantic.VERSION}")
    print(f"  requests: {requests.__version__}")

    response = requests.Response()
    response._content = b"not valid JSON"
    print(f"  result  : {parse_response(response)}")
PYTHON

sudo chown -R alice:alice "${ALICE_V1}"

#
# Show the content of main.py
sudo cat "${ALICE_MAIN}"

# Alice FastAPI project 1
# Uses requests.exceptions.JSONDecodeError, available since Requests 2.27.
import sys

import fastapi
import requests
from fastapi import FastAPI
from requests.exceptions import JSONDecodeError

app = FastAPI(title="alice-fastapi-v1")


def parse_response(response):
    try:
        return response.json()
    # unsupported by "requests==2.25.1"
    except JSONDecodeError:
        return {"error": "The service did not return JSON"}


@app.get("/")
def root():
    response = requests.Response()
    response._content = b"not valid JSON"

    return {
        "project": "alice-fastapi-v1",
        "fastapi": fastapi.__version__,
        "requests": requests.__version__,
        "parsed_response": parse_response(response),
    }


if __name__ == "__main__":
    print("Alice FastAPI v1")
    print(f"  Python  : {sys.executable}")
    print(f"  fastapi : {fastapi.__version__}")
    print(f"  requests: {requests.__version__}")

    response = requests.Response()
    r

### Alice FastAPI v2

The modern project: a current FastAPI and **Pydantic v2**, whose rewrite brought a much faster validation core and a new API — `model_dump()` replaces v1's `dict()`.

That one method name splits her world in two. `model_dump()` does not exist in Pydantic v1, and `BaseSettings` no longer exists in Pydantic v2. There is no version in between.

> ⚠️ **The second collision.** Same person, same account, same home directory. No users, permissions or `sudo` involved — two folders simply need two incompatible versions, and `pip install --user` has exactly one place to put a library.

Apart from the Pydantic version, the code mirrors project one.

In [ ]:
%%bash
set -euo pipefail

ALICE_V2="/home/alice/projects/fastapi-v2"
ALICE_MAIN="${ALICE_V2}/main.py"

sudo mkdir -p "${ALICE_V2}"

sudo tee "${ALICE_MAIN}" > /dev/null <<'PYTHON'
# Alice FastAPI project 2
#   requires pydantic v2  -> BaseModel.model_dump()
#   requires requests>=2.27 -> requests.exceptions.JSONDecodeError
import sys

import fastapi
import pydantic
import requests
from fastapi import FastAPI
from pydantic import BaseModel

# unsupported by "requests==2.25.1"
from requests.exceptions import JSONDecodeError


class Project(BaseModel):
    name: str
    version: str


app = FastAPI(title="alice-fastapi-v2")


def parse_response(response):
    try:
        return response.json()
    except JSONDecodeError:
        return {"error": "The service did not return JSON"}


@app.get("/")
def root():
    project = Project(name="alice-fastapi-v2", version=fastapi.__version__)
    response = requests.Response()
    response._content = b"not valid JSON"

    return {
        # model_dump() does not exist in pydantic v1
        "project": project.model_dump(),
        "fastapi": fastapi.__version__,
        "pydantic": pydantic.VERSION,
        "requests": requests.__version__,
        "parsed_response": parse_response(response),
    }


if __name__ == "__main__":
    print("Alice FastAPI v2")
    print(f"  Python  : {sys.executable}")
    print(f"  fastapi : {fastapi.__version__}")
    print(f"  pydantic: {pydantic.VERSION}")
    print(f"  requests: {requests.__version__}")

    project = Project(name="alice-fastapi-v2", version=fastapi.__version__)
    print(f"  model   : {project.model_dump()}")

    response = requests.Response()
    response._content = b"not valid JSON"
    print(f"  result  : {parse_response(response)}")
PYTHON

sudo chown -R alice:alice "${ALICE_V2}"

#
# Show the content of main.py
sudo cat "${ALICE_MAIN}"

# Alice FastAPI project 2
# Uses Pydantic 2 and requests.exceptions.JSONDecodeError.
import sys

import fastapi
import requests
from fastapi import FastAPI
from pydantic import BaseModel
from requests.exceptions import JSONDecodeError

app = FastAPI(title="alice-fastapi-v2")


class Project(BaseModel):
    name: str
    version: str


def parse_response(response):
    try:
        return response.json()
    except JSONDecodeError:
        return {"error": "The service did not return JSON"}


@app.get("/")
def root():
    project = Project(name="alice-fastapi-v2", version=fastapi.__version__)
    response = requests.Response()
    response._content = b"not valid JSON"

    return {
        "project": project.model_dump(),
        "fastapi": fastapi.__version__,
        "requests": requests.__version__,
        "parsed_response": parse_response(response),
    }


if __name__ == "__main__":
    print("Alice FastAPI v2")
    print(f"  Python  : {sys.executable}")
    print(f"  fastapi : {f

---

## Act V — Hitting the Walls

### Wall 1 — One Interpreter, Two Users

#### Bob's Application Works

> **Alice:** Bob, my app won't even start.
>
> **Bob:** Works on my machine. Literally — the same machine.

Both run the same binary, and that binary reads the same system directories, so `import requests` opens *the same file* for both of them. Bob asks for 2.25.1 and gets it.

In [26]:
%%bash
set -euo pipefail
python3 /home/bob/projects/legacy-project/main.py --domain "https://example.com"

Bob Legacy Application
  Python  : /usr/bin/python3
  requests: 2.25.1
  request : GET https://example.com


#### Alice's Application Fails

Same server, same interpreter, same `requests` — different code. `sudo -u alice -H` runs the command as her: `-u` picks the user, `-H` sets `HOME` so that her user site is the one that counts.

Read the last line of the traceback: Python found `requests`, opened it, and the class was not inside. Nothing is broken or misconfigured — the library is simply the wrong age, and Alice cannot change that.

In [ ]:
%%bash
sudo -u alice -H python3 /home/alice/projects/fastapi-v1/main.py || true

Alice FastAPI v1
  Python  : /usr/bin/python3
  fastapi : 0.68.2
  requests: 2.32.3
  result  : {'error': 'The service did not return JSON'}


### Knocking Down Wall 1 — One Copy per User

#### Bob Moves Into His Own User Site

> **Alice:** Could you take `requests` off the system Python, and let each of us keep our own copy?
>
> **Bob:** ...that's allowed?

It is. The shared copy is removed, and each user installs the version their own code needs into their own `~/.local`. Once nothing is shared, nothing has to be agreed on.

Bob gets his pinned 2.25.1 straight back — this time where it affects nobody but him. The uninstall needs `--break-system-packages` too: *removing* something from a distribution-managed interpreter is just as much of an intervention as adding it.

In [1]:
%%bash
set -euo pipefail
echo "Remove the shared Requests installation from the system interpreter"
sudo -H python3 -m pip uninstall --break-system-packages --yes requests

echo
echo "Install Bob's required Requests version in his own user site"
python3 -m pip install --user "requests==2.25.1"

Remove the shared Requests installation from the system interpreter



Install Bob's required Requests version in his own user site


In [ ]:
%%bash
set -euo pipefail
echo "Where does Bob's requests come from now?"
python3 -c "import requests; print(f'  requests {requests.__version__} loaded from {requests.__file__}')"

echo
echo "Run Bob's legacy application"
python3 /home/bob/projects/legacy-project/main.py

Run Bob's legacy application
Bob Legacy Application
  Python  : /usr/bin/python3
  requests: 2.25.1
  request : GET https://example.com


#### Alice Installs Her Own Version

Alice installs `requests==2.32.3` into her own user site. Because `~/.local` sits before the system directories on the import path, her interpreter picks up her copy while Bob's picks up his — out of the very same `/usr/bin/python3`.

Wall one is down. But note the *scope* of the fix: `--user` separates **people**, not **projects**.

In [2]:
%%bash
set -euo pipefail

echo "Install Alice's required Requests version in her user site"
sudo -u alice -H python3 -m pip install --user --upgrade "requests==2.32.3"

Install Alice's required Requests version in her user site


In [43]:
%%bash
set -euo pipefail
echo "Run Alice's FastAPI v1 application"
sudo -u alice -H python3 /home/alice/projects/fastapi-v1/main.py

Run Alice's FastAPI v1 application
Alice FastAPI v1
  Python  : /usr/bin/python3
  fastapi : 0.111.1
  requests: 2.32.3
  result  : {'error': 'The service did not return JSON'}


> ✅ **Wall 1 is down.** The shared copy of `requests` is gone. Bob imports 2.25.1 from `/home/bob/.local`, Alice imports 2.32.3 from `/home/alice/.local`, and neither of them can affect the other any more.
>
> ⚠️ **But:** Alice's *two* projects still share the one `~/.local` directory between them.

### Wall 2 — One User, Two Projects

#### Round 1 — Pydantic v1: Project One Works, Project Two Fails

Alice has exactly one `~/.local` and two projects that need different Pydantic versions. Whatever she installs there, one of them is wrong.

The older set goes in first: **project one runs**, and **project two fails**, because `model_dump()` does not exist in Pydantic v1.

In [ ]:
%%bash
set -euo pipefail
echo "Step 1: install the older FastAPI and Pydantic in Alice's shared user site"
sudo -u alice -H python3 -m pip install --user --break-system-packages --upgrade \
    "fastapi==0.68.2" "pydantic<2"

Install the older FastAPI and Pydantic versions in Alice's shared user site


In [ ]:
%%bash
set -euo pipefail
echo "Step 1: alice-fastapi-v1 with the shared user environment -- works"
sudo -u alice -H python3 /home/alice/projects/fastapi-v1/main.py

Run alice-fastapi-v1 with the shared user environment
Alice FastAPI v1
  Python  : /usr/bin/python3
  fastapi : 0.68.2
  requests: 2.32.3
  result  : {'error': 'The service did not return JSON'}


In [ ]:
%%bash
set -euo pipefail
echo "Step 2: alice-fastapi-v2 with the same shared user environment -- fails"
sudo -u alice -H python3 /home/alice/projects/fastapi-v2/main.py || true

Run alice-fastapi-v2 with the same shared user environment
Alice FastAPI v2
  Python  : /usr/bin/python3
  fastapi : 0.68.2
  requests: 2.32.3


Traceback (most recent call last):
  File "/home/alice/projects/fastapi-v2/main.py", line 47, in <module>
    print(f"  model   : {project.model_dump()}")
AttributeError: 'Project' object has no attribute 'model_dump'


#### Round 2 — Pydantic v2: Project Two Works, Project One Fails

> **Alice:** Right, I'll just upgrade. That's usually the answer.

Pydantic v2 and a current FastAPI go into the same `~/.local`. With only one directory, an upgrade is not an addition — it is a **replacement** for every project that reads from it.

In [ ]:
%%bash
set -euo pipefail
echo "Step 3: upgrade Alice's shared user site to the newer FastAPI and Pydantic"
sudo -u alice -H python3 -m pip install --user --break-system-packages --upgrade \
    "fastapi==0.111.1" "pydantic>=2"

In [ ]:
%%bash
set -euo pipefail
echo "Step 3: alice-fastapi-v2 with the upgraded shared user environment -- works"
sudo -u alice -H python3 /home/alice/projects/fastapi-v2/main.py

Project two is happy. Project one — which worked ninety seconds ago, and whose source nobody touched — is broken, because `BaseSettings` was removed in the version project two demanded.

That is what makes shared environments exhausting: the file that breaks is not the file you changed. No ordering of these commands escapes it. `--user` gave Alice a private environment; she needs *two*.

In [ ]:
%%bash
set -euo pipefail
echo "Step 4: alice-fastapi-v1 with the upgraded shared user environment -- now broken"
sudo -u alice -H python3 /home/alice/projects/fastapi-v1/main.py || true

### Knocking Down Wall 2 — One Environment per Project

#### Create One Environment per Project

> **Alice:** I need each project to have its own packages.
>
> **Bob:** So you want three Pythons on my server now?
>
> **Alice:** No. One Python, three package folders.

A **virtual environment** is a directory with its own `bin/python` and its own `site-packages`. It does not copy the interpreter — it links back to `/usr/bin/python3` and only changes the answer to one question: *where do I look for packages?* One command creates it:

```bash
python3 -m venv <directory>
```

It comes from `python3-venv`, which Bob installed on day one and never used.

Because each environment carries its own `site-packages`, "which Pydantic is installed?" stops being a property of the machine or of the user and becomes a property of the **project** — where it belonged all along.

The two cells below are identical except for the pinned versions. That is the point: the projects no longer know the other one exists. And no `--break-system-packages` anywhere, because nothing distribution-managed is touched.

In [ ]:
%%bash
set -euo pipefail
echo "Create the virtual environment for alice-fastapi-v1"
sudo -u alice -H python3 -m venv /home/alice/projects/fastapi-v1/.venv
echo "Upgrade pip inside alice-fastapi-v1/.venv"
sudo -u alice -H /home/alice/projects/fastapi-v1/.venv/bin/python -m pip install --quiet --upgrade pip
echo "Install project dependencies for alice-fastapi-v1"
sudo -u alice -H /home/alice/projects/fastapi-v1/.venv/bin/python -m pip install --quiet \
    "fastapi==0.68.2" "pydantic<2" "requests==2.32.3"

Create the virtual environment for alice-fastapi-v1
Upgrade pip inside alice-fastapi-v1/.venv
Install project dependencies for alice-fastapi-v1


In [ ]:
%%bash
set -euo pipefail
echo "Create the virtual environment for alice-fastapi-v2"
sudo -u alice -H python3 -m venv /home/alice/projects/fastapi-v2/.venv
echo "Upgrade pip inside alice-fastapi-v2/.venv"
sudo -u alice -H /home/alice/projects/fastapi-v2/.venv/bin/python -m pip install --quiet --upgrade pip
echo "Install project dependencies for alice-fastapi-v2"
sudo -u alice -H /home/alice/projects/fastapi-v2/.venv/bin/python -m pip install --quiet \
    "fastapi==0.111.1" "pydantic>=2" "requests==2.32.3"

Create the virtual environment for alice-fastapi-v2
Upgrade pip inside alice-fastapi-v2/.venv
Install project dependencies for alice-fastapi-v2


#### The Final State — Three Projects, Three Answers

All three run side by side, each reporting the interpreter that executed it and the versions it imported:

- **Bob's tool** on `/usr/bin/python3`, with `requests` 2.25.1 from his user site and `psutil` from APT.
- **Project one** on `fastapi-v1/.venv/bin/python`, with FastAPI 0.68.2 and Pydantic v1.
- **Project two** on `fastapi-v2/.venv/bin/python`, with FastAPI 0.111.1 and Pydantic v2.

Three incompatible dependency sets, one server, one interpreter underneath them all — and nobody needs permission from anybody.

> 💡 A virtual environment starts empty and does **not** see the APT directory, so `psutil` is not importable inside Alice's environments unless she installs it there. That is a feature — it is what makes an environment reproducible — and it is why Bob's tool, which depends on a distribution package, legitimately stays on the system interpreter.

In [ ]:
%%bash
set -euo pipefail
echo 'Bob Legacy (system interpreter + user site)'
echo '-------------------------------------------'
python3 /home/bob/projects/legacy-project/main.py
echo
echo 'Alice FastAPI v1 (own virtual environment)'
echo '-------------------------------------------'
sudo -u alice -H /home/alice/projects/fastapi-v1/.venv/bin/python /home/alice/projects/fastapi-v1/main.py
echo
echo 'Alice FastAPI v2 (own virtual environment)'
echo '-------------------------------------------'
sudo -u alice -H /home/alice/projects/fastapi-v2/.venv/bin/python /home/alice/projects/fastapi-v2/main.py

Bob Legacy
----------
Bob Legacy Application
  Python  : /usr/bin/python3
  requests: 2.25.1
  request : GET https://example.com

Alice FastAPI V1
----------------
Alice FastAPI v1
  Python  : /home/alice/projects/fastapi-v1/.venv/bin/python
  fastapi : 0.68.2
  requests: 2.32.3
  result  : {'error': 'The service did not return JSON'}

Alice FastAPI V2
----------------
Alice FastAPI v2
  Python  : /home/alice/projects/fastapi-v2/.venv/bin/python
  fastapi : 0.111.1
  requests: 2.32.3
  model   : {'name': 'alice-fastapi-v2', 'version': '0.111.1'}
  result  : {'error': 'The service did not return JSON'}


---

## Epilogue

> **Bob:** So I never needed to install `requests` for everybody.
>
> **Alice:** You never needed to install *any* application library for everybody.
>
> **Bob:** And this venv thing was on the machine the whole time?
>
> **Alice:** Since day one. `python3-venv`, right there in your own package list.
>
> **Bob:** ...fine. I'll put it in the runbook.

### Where a Package Can Live

| Location | Installed with | Affects | Use it for |
| --- | --- | --- | --- |
| `/usr/lib/python3/dist-packages` | `apt install python3-…` | everyone | tools that belong to the OS and are patched by the distribution, like `python3-psutil` |
| `/usr/local/lib/python3.X/dist-packages` | `sudo pip install` | everyone | almost nothing — this is where Bob's morning went wrong |
| `~/.local/lib/python3.X/site-packages` | `pip install --user` | one user, all their projects | small personal command-line tools |
| `<project>/.venv/lib/python3.X/site-packages` | `python3 -m venv` | one project | **application dependencies — the default answer** |

### Three Rules

1. **Dependencies belong to the application, not to the machine.** If one project needs a library, install it in that project's environment.
2. **`sudo pip install` is a configuration change to a shared system**, made on behalf of every user who will ever log in.
3. **`--break-system-packages` is a warning, not a step.** Whenever you need it, ask whether this should be a virtual environment instead. It almost always should.